# Netflix-Style Movie Recommendation System
### MovieLens 1M Dataset | K-Means · FP-Growth · Logistic Regression · Decision Tree · Random Forest

In [ ]:
%%capture
!pip install mlxtend

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                              silhouette_score, accuracy_score, f1_score)
from sklearn.metrics.pairwise import cosine_similarity
import scipy.stats as stats

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH    = '/content/drive/MyDrive/Netflix_ML_Project/data/'
MOVIES_PATH  = BASE_PATH + 'movies.dat'
RATINGS_PATH = BASE_PATH + 'ratings.dat'
USERS_PATH   = BASE_PATH + 'users.dat'

## Phase 1 — Data Loading & Preprocessing

In [ ]:
# .dat files use '::' as separator and require latin-1 encoding for special characters
movies = pd.read_csv(MOVIES_PATH, sep='::', engine='python',
                     names=['movie_id', 'title', 'genres'], encoding='latin-1')

ratings = pd.read_csv(RATINGS_PATH, sep='::', engine='python',
                      names=['user_id', 'movie_id', 'rating', 'timestamp'], encoding='latin-1')

users = pd.read_csv(USERS_PATH, sep='::', engine='python',
                    names=['user_id', 'gender', 'age', 'occupation', 'zip_code'], encoding='latin-1')

print(f"movies  : {movies.shape}")
print(f"ratings : {ratings.shape}")
print(f"users   : {users.shape}")

In [ ]:
# Single master dataframe joining all three sources
df = ratings.merge(movies, on='movie_id').merge(users, on='user_id')

print(f"Master dataframe shape : {df.shape}")
print(f"Missing values         : {df.isnull().sum().sum()}")
df.head()

In [ ]:
# Drop timestamp — not used in any model
df.drop(columns=['timestamp'], inplace=True)

# Extract release year from title string  e.g. 'Toy Story (1995)' -> 1995
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')

# Clean title by removing year suffix
movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)
movies['genres_list'] = movies['genres'].str.split('|')

print("Cleaning complete.")
movies[['title_clean', 'genres', 'year']].head(10)

In [ ]:
# Age is stored as bracket codes per the dataset README
age_map = {1: 'Under 18', 18: '18-24', 25: '25-34', 35: '35-44',
           45: '45-49', 50: '50-55', 56: '56+'}

occupation_map = {
    0: 'Other', 1: 'Academic/Educator', 2: 'Artist', 3: 'Clerical/Admin',
    4: 'College/Grad Student', 5: 'Customer Service', 6: 'Doctor/Health Care',
    7: 'Executive/Managerial', 8: 'Farmer', 9: 'Homemaker', 10: 'K-12 Student',
    11: 'Lawyer', 12: 'Programmer', 13: 'Retired', 14: 'Sales/Marketing',
    15: 'Scientist', 16: 'Self-employed', 17: 'Technician/Engineer',
    18: 'Tradesman/Craftsman', 19: 'Unemployed', 20: 'Writer'
}

users['age_label']        = users['age'].map(age_map)
users['occupation_label'] = users['occupation'].map(occupation_map)

users[['user_id', 'age', 'age_label', 'occupation', 'occupation_label']].head()

## Phase 2 — Exploratory Data Analysis

In [ ]:
# Verify no duplicate movie IDs or user-movie rating pairs exist
dup_movies  = movies.duplicated(subset=['movie_id']).sum()
dup_ratings = ratings.duplicated(subset=['user_id', 'movie_id']).sum()
print(f"Duplicate movie IDs  : {dup_movies}")
print(f"Duplicate ratings    : {dup_ratings}")

movies.drop_duplicates(subset=['movie_id'], inplace=True)
ratings.drop_duplicates(subset=['user_id', 'movie_id'], inplace=True)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='rating', data=ratings, palette='viridis')
plt.title('Distribution of Movie Ratings', fontsize=14, fontweight='bold')
plt.xlabel('Rating (1 to 5)')
plt.ylabel('Count')
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/rating_distribution.png', dpi=150)
plt.show()

In [ ]:
top_rated = (ratings.groupby('movie_id')['rating']
             .count().reset_index()
             .rename(columns={'rating': 'num_ratings'})
             .merge(movies[['movie_id', 'title_clean']], on='movie_id')
             .sort_values('num_ratings', ascending=False).head(10))

plt.figure(figsize=(10, 6))
sns.barplot(x='num_ratings', y='title_clean', data=top_rated, palette='magma')
plt.title('Top 10 Most Rated Movies', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings')
plt.ylabel('')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/top_rated_movies.png', dpi=150)
plt.show()

In [ ]:
# Minimum 500 ratings required to avoid small-sample bias
top_highest = (ratings.groupby('movie_id')
               .agg(avg_rating=('rating', 'mean'), num_ratings=('rating', 'count'))
               .reset_index()
               .query('num_ratings >= 500')
               .merge(movies[['movie_id', 'title_clean']], on='movie_id')
               .sort_values('avg_rating', ascending=False).head(10))

plt.figure(figsize=(10, 6))
sns.barplot(x='avg_rating', y='title_clean', data=top_highest, palette='coolwarm')
plt.title('Top 10 Highest Rated Movies (min 500 ratings)', fontsize=14, fontweight='bold')
plt.xlabel('Average Rating')
plt.ylabel('')
plt.xlim(3.5, 5.0)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/highest_rated_movies.png', dpi=150)
plt.show()

In [ ]:
genre_exploded = movies.copy()
genre_exploded['genres_list'] = genre_exploded['genres'].str.split('|')
genre_exploded = genre_exploded.explode('genres_list')

genre_counts = genre_exploded['genres_list'].value_counts().reset_index()
genre_counts.columns = ['genre', 'count']

plt.figure(figsize=(12, 6))
sns.barplot(x='count', y='genre', data=genre_counts, palette='Set2')
plt.title('Movie Genre Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Movies')
plt.ylabel('')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/genre_distribution.png', dpi=150)
plt.show()

In [ ]:
gender_counts = users['gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']
gender_counts['gender'] = gender_counts['gender'].map({'M': 'Male', 'F': 'Female'})

plt.figure(figsize=(6, 6))
plt.pie(gender_counts['count'], labels=gender_counts['gender'],
        autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
        startangle=90, textprops={'fontsize': 13})
plt.title('User Gender Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/gender_distribution.png', dpi=150)
plt.show()

In [ ]:
age_order  = ['Under 18', '18-24', '25-34', '35-44', '45-49', '50-55', '56+']
age_counts = users['age_label'].value_counts().reset_index()
age_counts.columns = ['age_label', 'count']
age_counts = age_counts.set_index('age_label').reindex(age_order).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(x='age_label', y='count', data=age_counts, palette='Blues_d', order=age_order)
plt.title('User Age Group Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Number of Users')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/age_distribution.png', dpi=150)
plt.show()

In [ ]:
occ_counts = users['occupation_label'].value_counts().reset_index()
occ_counts.columns = ['occupation_label', 'count']

plt.figure(figsize=(12, 7))
sns.barplot(x='count', y='occupation_label', data=occ_counts, palette='tab20')
plt.title('User Occupation Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Users')
plt.ylabel('')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/occupation_distribution.png', dpi=150)
plt.show()

In [ ]:
genre_ratings = ratings.merge(movies[['movie_id', 'genres']], on='movie_id')
genre_ratings['genres_list'] = genre_ratings['genres'].str.split('|')
genre_ratings_exploded = genre_ratings.explode('genres_list')

avg_genre_rating = (genre_ratings_exploded
                    .groupby('genres_list')['rating'].mean()
                    .sort_values(ascending=False).reset_index()
                    .rename(columns={'genres_list': 'genre', 'rating': 'avg_rating'}))

plt.figure(figsize=(12, 6))
sns.barplot(x='avg_rating', y='genre', data=avg_genre_rating, palette='RdYlGn')
plt.title('Average Rating by Genre', fontsize=14, fontweight='bold')
plt.xlabel('Average Rating')
plt.ylabel('')
plt.xlim(3.0, 4.5)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/avg_rating_by_genre.png', dpi=150)
plt.show()

In [ ]:
# timestamp is sourced from ratings directly since it was dropped from df
ratings_time = ratings.copy()
ratings_time['year_rated'] = pd.to_datetime(ratings_time['timestamp'], unit='s').dt.year

ratings_per_year = (ratings_time.groupby('year_rated')['rating']
                    .count().reset_index()
                    .rename(columns={'rating': 'num_ratings'}))

plt.figure(figsize=(10, 5))
sns.lineplot(x='year_rated', y='num_ratings', data=ratings_per_year,
             marker='o', color='royalblue', linewidth=2.5)
plt.title('Rating Activity Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Number of Ratings')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/ratings_over_time.png', dpi=150)
plt.show()

## Phase 3 — K-Means Movie Clustering

In [ ]:
# Two confirmed data entry errors in the dataset (noted in README)
movies.loc[movies['title_clean'].str.contains('Godfather: Part II', case=False), 'genres'] = 'Crime|Drama'
movies.loc[movies['title_clean'].str.contains('Blade Runner', case=False), 'genres'] = 'Sci-Fi|Thriller'

print("Genre corrections applied.")
print(movies[movies['title_clean'].str.contains('Godfather|Blade Runner', case=False)][['title_clean', 'genres']])

In [ ]:
# One-hot encode all 18 genres — each genre becomes a binary feature column
genres_dummies = movies['genres'].str.get_dummies(sep='|')
print(f"Genre matrix shape : {genres_dummies.shape}")
print(f"Genres             : {list(genres_dummies.columns)}")

In [ ]:
# Compute per-movie rating statistics and merge with genre features
movie_stats = (ratings.groupby('movie_id')
               .agg(avg_rating=('rating', 'mean'), num_ratings=('rating', 'count'))
               .reset_index())

movies_cluster = movies[['movie_id', 'title_clean', 'year']].merge(movie_stats, on='movie_id')
movies_cluster = movies_cluster.merge(
    genres_dummies, left_index=False, right_index=True,
    left_on=movies_cluster.index, right_on=genres_dummies.index
).drop(columns=['key_0'], errors='ignore')

print(f"Clustering feature matrix shape : {movies_cluster.shape}")
movies_cluster.head()

In [ ]:
# Scale features so avg_rating and num_ratings do not dominate the genre binary columns
cluster_feature_cols = list(genres_dummies.columns) + ['avg_rating', 'num_ratings']
X_movies = movies_cluster[cluster_feature_cols].copy()

scaler = StandardScaler()
X_movies_scaled = scaler.fit_transform(X_movies)

print(f"Scaled feature matrix shape : {X_movies_scaled.shape}")

In [ ]:
# Elbow method — plot inertia across K=2 to K=10
inertia  = []
k_range  = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_movies_scaled)
    inertia.append(km.inertia_)
    print(f"  K={k}  Inertia: {km.inertia_:.2f}")

plt.figure(figsize=(9, 5))
plt.plot(k_range, inertia, marker='o', color='royalblue', linewidth=2.5)
plt.title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/elbow_curve.png', dpi=150)
plt.show()

In [ ]:
# Silhouette score provides a second validation signal for optimal K
sil_scores = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_movies_scaled)
    score  = silhouette_score(X_movies_scaled, labels)
    sil_scores.append(score)
    print(f"  K={k}  Silhouette Score: {score:.4f}")

plt.figure(figsize=(9, 5))
plt.plot(k_range, sil_scores, marker='s', color='coral', linewidth=2.5)
plt.title('Silhouette Scores by K', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/silhouette_scores.png', dpi=150)
plt.show()

In [ ]:
# K=5 selected based on silhouette score peak and interpretability
OPTIMAL_K = 5

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
movies_cluster['cluster'] = kmeans.fit_predict(X_movies_scaled)

print(f"K-Means trained with K={OPTIMAL_K}")
print(movies_cluster['cluster'].value_counts().sort_index())

In [ ]:
# PCA reduces the 20-dimensional feature space to 2D for visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_movies_scaled)

movies_cluster['pca_x'] = X_pca[:, 0]
movies_cluster['pca_y'] = X_pca[:, 1]

cluster_labels = {
    0: 'Drama & Comedy',
    1: 'Family, Animation & Musicals',
    2: 'Thrillers & Crime Mysteries',
    3: 'Action & Adventure',
    4: 'Documentaries'
}

plt.figure(figsize=(12, 7))
palette = sns.color_palette('tab10', OPTIMAL_K)

for cluster_id in range(OPTIMAL_K):
    mask = movies_cluster['cluster'] == cluster_id
    plt.scatter(movies_cluster.loc[mask, 'pca_x'],
                movies_cluster.loc[mask, 'pca_y'],
                label=cluster_labels[cluster_id], alpha=0.6, s=40,
                color=palette[cluster_id])

plt.title('Movie Clusters — PCA 2D Projection', fontsize=14, fontweight='bold')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster', fontsize=10, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/movie_clusters_pca.png', dpi=150)
plt.show()

In [ ]:
# Mean genre composition and rating stats per cluster
genre_cols = list(genres_dummies.columns)
cluster_profiles = movies_cluster.groupby('cluster')[genre_cols + ['avg_rating', 'num_ratings']].mean().round(3)
print("Cluster Profiles (mean values):")
print(cluster_profiles.to_string())

In [ ]:
# Top 5 most-rated titles in each cluster confirm the cluster identities
print("Sample Movies per Cluster")
print("=" * 60)
for cluster_id in range(OPTIMAL_K):
    cluster_movies = (movies_cluster[movies_cluster['cluster'] == cluster_id]
                      .sort_values('num_ratings', ascending=False)
                      .head(5)['title_clean'].tolist())
    print(f"\nCluster {cluster_id} — {cluster_labels[cluster_id]}:")
    for movie in cluster_movies:
        print(f"  {movie}")

In [ ]:
cluster_names = {
    0: 'Drama & Comedy',
    1: 'Family, Animation & Musicals',
    2: 'Thrillers & Crime Mysteries',
    3: 'Action & Adventure',
    4: 'Documentaries'
}

movies_cluster['cluster_name'] = movies_cluster['cluster'].map(cluster_names)
print(movies_cluster['cluster_name'].value_counts())

## Phase 4 — FP-Growth Co-watch Pattern Mining

In [ ]:
# Each user's liked movies (rating >= 3.5) form one transaction basket
liked_ratings = ratings[ratings['rating'] >= 3.5]

transactions = (liked_ratings.merge(movies[['movie_id', 'title_clean']], on='movie_id')
                .groupby('user_id')['title_clean'].apply(list).reset_index())

print(f"Total users with liked movies : {transactions.shape[0]:,}")
print(f"Sample basket (User 1)        : {transactions.iloc[0]['title_clean'][:5]}")

In [ ]:
# Encode transactions as a boolean user-item matrix
te       = TransactionEncoder()
te_array = te.fit_transform(transactions['title_clean'])
te_df    = pd.DataFrame(te_array, columns=te.columns_)

print(f"Transaction matrix shape : {te_df.shape}  (users x movies)")
memory_mb = te_df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage             : {memory_mb:.1f} MB")

In [ ]:
# Restrict to top 300 most-rated movies to keep FP-Growth tractable on Colab free
TOP_N_MOVIES = 300
MIN_SUPPORT  = 0.05

top_movies = (ratings.groupby('movie_id')['rating'].count().reset_index()
              .rename(columns={'rating': 'num_ratings'})
              .merge(movies[['movie_id', 'title_clean']], on='movie_id')
              .sort_values('num_ratings', ascending=False)
              .head(TOP_N_MOVIES)['title_clean'].tolist())

te_df_filtered = te_df[[col for col in top_movies if col in te_df.columns]]

print(f"Movies before filtering : {te_df.shape[1]:,}")
print(f"Movies after filtering  : {te_df_filtered.shape[1]:,}")
print(f"Filtered matrix memory  : {te_df_filtered.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# max_len=2 restricts itemsets to movie pairs, which is all we need for pairwise rules
frequent_itemsets = fpgrowth(te_df_filtered, min_support=MIN_SUPPORT,
                              use_colnames=True, max_len=2)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f"Frequent itemsets found : {frequent_itemsets.shape[0]:,}")
print("\nTop 10 Most Frequent Itemsets:")
print(frequent_itemsets.head(10).to_string())

In [ ]:
# Generate association rules with minimum confidence of 0.5
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)
rules = rules.sort_values('lift', ascending=False)

print(f"Association rules generated : {rules.shape[0]:,}")
print("\nTop 10 Rules by Lift:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string())

In [ ]:
# Bubble chart — size and colour both encode lift value
top_rules = rules.head(20).copy()
top_rules['antecedents_str'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents_str'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))

plt.figure(figsize=(12, 8))
scatter = plt.scatter(top_rules['support'], top_rules['confidence'],
                      c=top_rules['lift'], s=top_rules['lift'] * 80,
                      cmap='YlOrRd', alpha=0.7, edgecolors='black', linewidths=0.5)

for i, row in top_rules.iterrows():
    plt.annotate(row['antecedents_str'], (row['support'], row['confidence']),
                 fontsize=7, xytext=(5, 5), textcoords='offset points')

plt.colorbar(scatter, label='Lift')
plt.title('FP-Growth Association Rules  (Size & Colour = Lift)', fontsize=14, fontweight='bold')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/association_rules.png', dpi=150)
plt.show()

In [ ]:
def because_you_watched(movie_title, rules_df, top_n=5):
    """Return top_n co-watch recommendations for a given movie title using association rules."""
    mask = rules_df['antecedents'].apply(
        lambda x: any(movie_title.lower() in m.lower() for m in x))
    relevant_rules = rules_df[mask].head(top_n)

    if relevant_rules.empty:
        print(f"No rules found for '{movie_title}'. Try a more popular title.")
        return

    print(f"Because you watched '{movie_title}':")
    print("-" * 50)
    for _, row in relevant_rules.iterrows():
        consequent = ', '.join(list(row['consequents']))
        print(f"  {consequent}")
        print(f"    Confidence : {row['confidence']:.2%}  |  Lift : {row['lift']:.2f}")

because_you_watched('Silence of the Lambs', rules)
print()
because_you_watched('Toy Story', rules)
print()
because_you_watched('Matrix', rules)
print()
because_you_watched('Star Wars: Episode IV', rules)

In [ ]:
# Top 15 rules summary for reporting
summary = rules.head(15).copy()
summary['antecedents'] = summary['antecedents'].apply(lambda x: ', '.join(list(x)))
summary['consequents'] = summary['consequents'].apply(lambda x: ', '.join(list(x)))
summary = summary[['antecedents', 'consequents', 'support', 'confidence', 'lift']].round(3)

print("Top 15 Association Rules")
print("=" * 90)
print(summary.to_string(index=False))
print("\nNote: Support = % users watched both  |  Confidence = P(B|A)  |  Lift > 1 = meaningful")

## Phase 5 — Supervised Models

In [ ]:
# Ratings >= 3.5 are treated as a 'Like' (1), below as a 'Dislike' (0)
df['liked'] = (df['rating'] >= 3.5).astype(int)

print("Class distribution:")
print(df['liked'].value_counts())
print()
print("Class balance (%):")
print(df['liked'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# Merge cluster assignment and movie-level stats into master dataframe
df = df.merge(movies_cluster[['movie_id', 'cluster', 'cluster_name', 'avg_rating', 'num_ratings']],
              on='movie_id', how='left')

df['gender_encoded'] = (df['gender'] == 'M').astype(int)

# One-hot encode genres at the rating level
genre_features = movies[['movie_id', 'genres']].copy()
genre_features = genre_features.join(genre_features['genres'].str.get_dummies(sep='|'))
genre_features.drop(columns=['genres'], inplace=True)

df = df.merge(genre_features, on='movie_id', how='left')
print(f"Master dataframe shape : {df.shape}")

In [ ]:
# User-level personalisation features capture individual rating behaviour
user_avg   = ratings.groupby('user_id')['rating'].mean().reset_index()
user_count = ratings.groupby('user_id')['rating'].count().reset_index()
user_avg.columns   = ['user_id', 'user_avg_rating']
user_count.columns = ['user_id', 'user_num_ratings']

df = df.merge(user_avg, on='user_id', how='left')
df = df.merge(user_count, on='user_id', how='left')
print(df[['user_id', 'user_avg_rating', 'user_num_ratings']].head())

In [ ]:
feature_cols = [
    'age', 'gender_encoded', 'occupation',       # user demographics
    'user_avg_rating', 'user_num_ratings',        # user behaviour
    'avg_rating', 'num_ratings', 'cluster',       # movie-level features
    'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
    'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'  # genre flags
]

X = df[feature_cols].fillna(0)
y = df['liked']

print(f"Feature matrix : {X.shape}")
print(f"Target vector  : {y.shape}")

In [ ]:
# stratify=y ensures both splits retain the original 57.5/42.5 class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set : {X_train.shape[0]:,} rows")
print(f"Test set     : {X_test.shape[0]:,} rows")

In [ ]:
# Logistic Regression requires standardised features; tree-based models do not
scaler_lr      = StandardScaler()
X_train_scaled = scaler_lr.fit_transform(X_train)
X_test_scaled  = scaler_lr.transform(X_test)

In [ ]:
lr      = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

print("Logistic Regression — Classification Report")
print(classification_report(y_test, lr_pred, target_names=['Dislike', 'Like']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, display_labels=['Dislike', 'Like'],
                                         colorbar=False, ax=ax, cmap='Blues')
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/lr_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
dt = DecisionTreeClassifier(random_state=42, max_depth=6, min_samples_leaf=50)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("Decision Tree — Classification Report")
print(classification_report(y_test, dt_pred, target_names=['Dislike', 'Like']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, dt_pred, display_labels=['Dislike', 'Like'],
                                         colorbar=False, ax=ax, cmap='Greens')
ax.set_title('Decision Tree — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/dt_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Visualise the top 3 levels of the decision tree for interpretability
plt.figure(figsize=(25, 10))
plot_tree(dt, feature_names=feature_cols, class_names=['Dislike', 'Like'],
          filled=True, rounded=True, fontsize=9, max_depth=3)
plt.title('Decision Tree — Top 3 Levels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                            max_depth=10, min_samples_leaf=50, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest — Classification Report")
print(classification_report(y_test, rf_pred, target_names=['Dislike', 'Like']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred, display_labels=['Dislike', 'Like'],
                                         colorbar=False, ax=ax, cmap='Oranges')
ax.set_title('Random Forest — Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/rf_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
importances = pd.DataFrame({'feature': feature_cols,
                            'importance': rf.feature_importances_}
                           ).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importances, palette='viridis')
plt.title('Random Forest — Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/feature_importance.png', dpi=150)
plt.show()

In [ ]:
results = pd.DataFrame({
    'Model'             : ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy'          : [accuracy_score(y_test, lr_pred), accuracy_score(y_test, dt_pred), accuracy_score(y_test, rf_pred)],
    'F1 Score (Like)'   : [f1_score(y_test, lr_pred), f1_score(y_test, dt_pred), f1_score(y_test, rf_pred)],
    'F1 Score (Dislike)': [f1_score(y_test, lr_pred, pos_label=0), f1_score(y_test, dt_pred, pos_label=0), f1_score(y_test, rf_pred, pos_label=0)],
    'Weighted F1'       : [f1_score(y_test, lr_pred, average='weighted'), f1_score(y_test, dt_pred, average='weighted'), f1_score(y_test, rf_pred, average='weighted')]
}).round(4)

print("Baseline Model Comparison")
print("=" * 75)
print(results.to_string(index=False))

In [ ]:
# RandomizedSearchCV with 3-fold CV and 10 iterations per model
param_dist_lr = {'C': stats.loguniform(0.01, 100), 'solver': ['lbfgs', 'saga'], 'penalty': ['l2']}
param_dist_dt = {'max_depth': [4, 6, 8, 10, 12], 'min_samples_leaf': [25, 50, 75, 100],
                 'criterion': ['gini', 'entropy'], 'max_features': ['sqrt', 'log2', None]}
param_dist_rf = {'n_estimators': [50, 100, 150], 'max_depth': [6, 8, 10, 12],
                 'min_samples_leaf': [25, 50, 75], 'max_features': ['sqrt', 'log2']}

random_search_lr = RandomizedSearchCV(LogisticRegression(random_state=42, max_iter=1000),
    param_dist_lr, n_iter=10, cv=3, scoring='f1_weighted', random_state=42, n_jobs=-1, verbose=1)
random_search_lr.fit(X_train_scaled, y_train)
print(f"Best LR params : {random_search_lr.best_params_}  |  CV F1: {random_search_lr.best_score_:.4f}")

random_search_dt = RandomizedSearchCV(DecisionTreeClassifier(random_state=42),
    param_dist_dt, n_iter=10, cv=3, scoring='f1_weighted', random_state=42, n_jobs=-1, verbose=1)
random_search_dt.fit(X_train, y_train)
print(f"Best DT params : {random_search_dt.best_params_}  |  CV F1: {random_search_dt.best_score_:.4f}")

random_search_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist_rf, n_iter=10, cv=3, scoring='f1_weighted', random_state=42, n_jobs=-1, verbose=1)
random_search_rf.fit(X_train, y_train)
print(f"Best RF params : {random_search_rf.best_params_}  |  CV F1: {random_search_rf.best_score_:.4f}")

In [ ]:
lr_tuned_pred = random_search_lr.best_estimator_.predict(X_test_scaled)
dt_tuned_pred = random_search_dt.best_estimator_.predict(X_test)
rf_tuned_pred = random_search_rf.best_estimator_.predict(X_test)

results_tuned = pd.DataFrame({
    'Model'      : ['LR Baseline', 'LR Tuned', 'DT Baseline', 'DT Tuned', 'RF Baseline', 'RF Tuned'],
    'Accuracy'   : [accuracy_score(y_test, p) for p in [lr_pred, lr_tuned_pred, dt_pred, dt_tuned_pred, rf_pred, rf_tuned_pred]],
    'Weighted F1': [f1_score(y_test, p, average='weighted') for p in [lr_pred, lr_tuned_pred, dt_pred, dt_tuned_pred, rf_pred, rf_tuned_pred]],
    'F1 Like'    : [f1_score(y_test, p) for p in [lr_pred, lr_tuned_pred, dt_pred, dt_tuned_pred, rf_pred, rf_tuned_pred]],
    'F1 Dislike' : [f1_score(y_test, p, pos_label=0) for p in [lr_pred, lr_tuned_pred, dt_pred, dt_tuned_pred, rf_pred, rf_tuned_pred]]
}).round(4)

print("Baseline vs Tuned Comparison")
print("=" * 75)
print(results_tuned.to_string(index=False))

In [ ]:
results_final = pd.DataFrame({
    'Model'             : ['Logistic Regression', 'Decision Tree (Tuned)', 'Random Forest (Tuned)'],
    'Accuracy'          : [accuracy_score(y_test, p) for p in [lr_tuned_pred, dt_tuned_pred, rf_tuned_pred]],
    'F1 Score (Like)'   : [f1_score(y_test, p) for p in [lr_tuned_pred, dt_tuned_pred, rf_tuned_pred]],
    'F1 Score (Dislike)': [f1_score(y_test, p, pos_label=0) for p in [lr_tuned_pred, dt_tuned_pred, rf_tuned_pred]],
    'Weighted F1'       : [f1_score(y_test, p, average='weighted') for p in [lr_tuned_pred, dt_tuned_pred, rf_tuned_pred]]
}).round(4)

print("Final Model Comparison (Tuned)")
print("=" * 70)
print(results_final.to_string(index=False))
print(f"\nBest model : {results_final.loc[results_final['Weighted F1'].idxmax(), 'Model']}")

In [ ]:
metrics = ['Accuracy', 'F1 Score (Like)', 'F1 Score (Dislike)', 'Weighted F1']
x, width = np.arange(len(metrics)), 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (label, color) in enumerate(zip(['Logistic Regression', 'Decision Tree (Tuned)', 'Random Forest (Tuned)'],
                                        ['steelblue', 'mediumseagreen', 'coral'])):
    bars = ax.bar(x + (i - 1) * width, results_final.iloc[i][metrics], width, label=label, color=color)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

ax.set_title('Final Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0.5, 0.85)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Netflix_ML_Project/model_comparison.png', dpi=150)
plt.show()

## Phase 6 — Recommendation Pipeline & Live Demo

In [ ]:
def find_similar_users(user_id, ratings_df, n_similar=20):
    """Compute cosine similarity between the target user and all others in the rating matrix."""
    user_item        = ratings_df.pivot_table(index='user_id', columns='movie_id', values='rating').fillna(0)
    user_item_sparse = csr_matrix(user_item.values)
    user_idx         = user_item.index.get_loc(user_id)
    similarities     = cosine_similarity(user_item_sparse[user_idx], user_item_sparse).flatten()
    similar_indices  = similarities.argsort()[::-1][1:n_similar + 1]
    return user_item.index[similar_indices].tolist(), similarities[similar_indices].tolist(), user_item


def recommend_movies(user_id, n_recommendations=10):
    """
    Generate personalised movie recommendations for a given user.

    Pipeline:
      1. Retrieve user profile and rating history
      2. Identify similar users via cosine similarity (User-Based CF)
      3. Collect candidate movies from similar users and FP-Growth rules
      4. Score candidates using the tuned Random Forest model
      5. Rank by a weighted combination of RF probability and CF signal
    """

    print(f"Recommendations for User {user_id}")
    print("-" * 60)

    user_info    = users[users['user_id'] == user_id].iloc[0]
    print(f"\nProfile   : {user_info['gender']} | {user_info['age_label']} | {user_info['occupation_label']}")

    user_ratings = df[df['user_id'] == user_id].copy()
    user_ratings = user_ratings.merge(movies[['movie_id', 'title_clean']], on='movie_id', how='left')
    liked_movies = user_ratings[user_ratings['liked'] == 1]
    watched      = set(user_ratings['title_clean'].tolist())

    print(f"Rated     : {len(user_ratings)}  |  Liked : {len(liked_movies)}  |  Avg rating : {user_ratings['rating'].mean():.2f}")

    top_liked = liked_movies.sort_values('rating', ascending=False).head(5)['title_clean'].tolist()
    print("\nTop 5 liked movies:")
    for m in top_liked:
        print(f"  {m}")

    # User-Based Collaborative Filtering
    similar_users, _, _ = find_similar_users(user_id, ratings, n_similar=20)
    similar_ratings = (ratings[ratings['user_id'].isin(similar_users)]
                       .merge(movies[['movie_id', 'title_clean']], on='movie_id', how='left'))

    cf_candidates = (similar_ratings[similar_ratings['rating'] >= 3.5]
                     .groupby('title_clean')['rating']
                     .agg(['mean', 'count']).reset_index()
                     .rename(columns={'mean': 'avg_sim_rating', 'count': 'sim_user_count'})
                     .sort_values('sim_user_count', ascending=False))
    cf_candidates = cf_candidates[~cf_candidates['title_clean'].isin(watched)]

    # FP-Growth candidates
    fp_candidates = set()
    for movie in top_liked[:5]:
        mask = rules['antecedents'].apply(lambda x: any(movie.lower() in m.lower() for m in x))
        for _, row in rules[mask].head(10).iterrows():
            fp_candidates.update(row['consequents'])
    fp_candidates -= watched

    print(f"\nCF candidates    : {len(cf_candidates)}")
    print(f"FP-Growth candidates : {len(fp_candidates)}")

    all_candidates = set(cf_candidates['title_clean'].tolist()) | fp_candidates
    if len(all_candidates) < n_recommendations:
        global_top     = set(movies_cluster.sort_values('avg_rating', ascending=False).head(100)['title_clean']) - watched
        all_candidates = all_candidates | global_top

    # Build feature matrix for scoring
    candidate_movie_data = movies_cluster[movies_cluster['title_clean'].isin(list(all_candidates))].copy()
    if candidate_movie_data.empty:
        print("No candidates found.")
        return None

    candidate_movie_data['age']              = user_info['age']
    candidate_movie_data['gender_encoded']   = int(user_info['gender'] == 'M')
    candidate_movie_data['occupation']       = user_info['occupation']
    candidate_movie_data['user_avg_rating']  = user_ratings['rating'].mean()
    candidate_movie_data['user_num_ratings'] = len(user_ratings)

    candidate_movie_data = candidate_movie_data.merge(
        cf_candidates[['title_clean', 'avg_sim_rating', 'sim_user_count']], on='title_clean', how='left')
    candidate_movie_data[['avg_sim_rating', 'sim_user_count']] = (
        candidate_movie_data[['avg_sim_rating', 'sim_user_count']].fillna(0))

    genre_data = (movies[movies['title_clean'].isin(list(all_candidates))][['title_clean', 'genres']].copy())
    genre_data = pd.concat([genre_data[['title_clean']], genre_data['genres'].str.get_dummies(sep='|')], axis=1)
    candidate_movie_data = candidate_movie_data.merge(genre_data, on='title_clean', how='left')
    candidate_movie_data = candidate_movie_data.drop_duplicates(subset=['title_clean'])

    for col in feature_cols:
        if col not in candidate_movie_data.columns:
            candidate_movie_data[col] = 0

    X_candidates = candidate_movie_data[feature_cols].fillna(0)
    like_proba   = random_search_rf.best_estimator_.predict_proba(X_candidates)[:, 1]

    candidate_movie_data['like_probability'] = like_proba
    max_sim = max(candidate_movie_data['sim_user_count'].max(), 1)
    candidate_movie_data['final_score'] = (
        0.6 * candidate_movie_data['like_probability'] +
        0.4 * (candidate_movie_data['sim_user_count'] / max_sim)
    )

    final_recs = candidate_movie_data.sort_values('final_score', ascending=False).head(n_recommendations)

    print(f"\nTop {n_recommendations} Recommendations:")
    print()
    for i, (_, row) in enumerate(final_recs.iterrows(), 1):
        print(f"  {i:2}. {row['title_clean']}")
        print(f"      Like probability  : {row['like_probability']:.2%}")
        print(f"      Similar users liked: {int(row['sim_user_count'])}")
        print(f"      Category          : {cluster_names.get(row['cluster'], 'Unknown')}")
    print("\n")
    return final_recs

In [ ]:
recs_user1 = recommend_movies(user_id=1)
recs_user2 = recommend_movies(user_id=50)
recs_user3 = recommend_movies(user_id=100)

In [ ]:
# Change USER_ID to any value between 1 and 6040 for the live presentation demo
USER_ID           = 42
N_RECOMMENDATIONS = 10

recs = recommend_movies(user_id=USER_ID, n_recommendations=N_RECOMMENDATIONS)

In [ ]:
def visualise_recommendations(recs, user_id):
    """Bar chart of recommendation scores for a given user."""
    if recs is None or recs.empty:
        print("No recommendations to display.")
        return

    plt.figure(figsize=(12, 7))
    colors = sns.color_palette('RdYlGn', len(recs))
    bars   = plt.barh(recs['title_clean'][::-1], recs['like_probability'][::-1],
                      color=colors, edgecolor='black', linewidth=0.5)

    for bar, prob in zip(bars, recs['like_probability'][::-1]):
        plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                 f'{prob:.1%}', va='center', fontsize=10, fontweight='bold')

    plt.title(f'Top Recommendations for User {user_id}\n(Scored by Random Forest Like Probability)',
              fontsize=13, fontweight='bold')
    plt.xlabel('Like Probability')
    plt.xlim(0, 1.1)
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Netflix_ML_Project/recommendations_user{user_id}.png', dpi=150)
    plt.show()

visualise_recommendations(recs_user1, user_id=1)
visualise_recommendations(recs_user2, user_id=50)
visualise_recommendations(recs_user3, user_id=100)